In [4]:
# %% ============================================================
# COLLECT analysis_CTNN_N{N}_omega_{omega}.json -> compact TeX tables + long CSVs
# Output: ../results/tables/texfiles (created if missing)
# ============================================================

from __future__ import annotations

import json
import math
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

# ---------------------------
# USER KNOBS (edit these)
# ---------------------------
IN_DIR  = Path("../results/tables/jsons")       # folder containing analysis_CTNN_N{N}_omega_{omega}.json
OUT_DIR = Path("../results/tables/texfiles")    # tables written here


# ---------------------------
# Helpers
# ---------------------------

def _safe_float(x: Any) -> float:
    try:
        if x is None:
            return float("nan")
        return float(x)
    except Exception:
        return float("nan")

def _safe_int(x: Any) -> int:
    try:
        if x is None:
            return -1
        return int(x)
    except Exception:
        return -1

def _get(d: dict[str, Any], *keys: str, default: Any = None) -> Any:
    cur: Any = d
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

def _as_list(x: Any) -> list[Any]:
    return x if isinstance(x, list) else []

def _fmt_omega_str(omega: float) -> str:
    if not np.isfinite(omega):
        return "--"
    if omega >= 0.1:
        return f"{omega:.3g}"
    s = f"{omega:.6f}".rstrip("0").rstrip(".")
    return s

def _zscore(delta: float, se_a: float, se_b: float) -> float:
    denom = math.sqrt(se_a*se_a + se_b*se_b)
    return float(delta/denom) if denom > 0 else float("nan")

def _k_threshold(k_list: list[int], v_list: list[float], frac_of_first: float) -> float:
    # smallest k such that v <= frac * v_first
    if not k_list or not v_list:
        return float("nan")
    v0 = float(v_list[0])
    if not np.isfinite(v0) or v0 == 0:
        return float("nan")
    thr = frac_of_first * v0
    for k, v in zip(k_list, v_list, strict=False):
        v = float(v)
        if np.isfinite(v) and v <= thr:
            return float(k)
    return float("nan")

def _short_columns(df: pd.DataFrame, mapping: dict[str, str]) -> pd.DataFrame:
    return df.rename(columns={k: v for k, v in mapping.items() if k in df.columns})

def _write_df(df: pd.DataFrame, out_csv: Path, out_tex: Path, caption: str, label: str, floatfmt: str = "%.4g"):
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False)

    # Column format: first col left, rest right (keeps tables readable)
    colfmt = "l" + "r" * max(len(df.columns)-1, 1)

    tex = df.to_latex(
        index=False,
        escape=True,
        na_rep="--",
        float_format=(lambda x: floatfmt % x),
        caption=caption,
        label=label,
        column_format=colfmt,
        longtable=False,
    )
    out_tex.parent.mkdir(parents=True, exist_ok=True)
    out_tex.write_text(tex, encoding="utf-8")


# ---------------------------
# Filename parsing (backup if JSON missing system fields)
# ---------------------------
# expected: analysis_CTNN_N20_omega_0.001.json
_RE = re.compile(r"analysis_CTNN_N(?P<N>\d+)_omega_(?P<omega>[0-9.]+)(?:\.json)?$")

def _parse_N_omega_from_name(name: str):
    m = _RE.match(name)
    if not m:
        return None, None
    return int(m.group("N")), float(m.group("omega"))


# ---------------------------
# Extraction
# ---------------------------

@dataclass
class Extracted:
    system_energy_mcmc: dict[str, Any]
    pinn_summary: dict[str, Any]
    backflow_summary: dict[str, Any]
    rho_weights_long: list[dict[str, Any]]
    energy_feat_corr_long: list[dict[str, Any]]
    dx_pc1_particle_long: list[dict[str, Any]]
    pinn_pc_ablation_long: list[dict[str, Any]]
    dx_pc_ablation_long: list[dict[str, Any]]


def _read_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def _extract_one(path: Path) -> Extracted:
    j = _read_json(path)

    # Prefer JSON system keys, fallback to filename
    sysN = _safe_int(_get(j, "system", "N"))
    omega = _safe_float(_get(j, "system", "omega"))
    if sysN < 0 or not np.isfinite(omega):
        fnN, fnw = _parse_N_omega_from_name(path.name)
        if sysN < 0 and fnN is not None:
            sysN = fnN
        if not np.isfinite(omega) and fnw is not None:
            omega = fnw
    d = _safe_int(_get(j, "system", "d", default=2))


    # ----- MCMC
    mcmc = _get(j, "mcmc", default={}) or {}
    method = str(mcmc.get("method", ""))
    step_sigma = _safe_float(mcmc.get("step_sigma"))
    burn_in = _safe_int(mcmc.get("burn_in"))
    mix_steps = _safe_int(mcmc.get("mix_steps"))
    target_accept = _safe_float(mcmc.get("target_accept"))

    acc_noBF = mcmc.get("acc_noBF", {}) if isinstance(mcmc.get("acc_noBF", {}), dict) else {}
    acc_BF = mcmc.get("acc_BF", {}) if isinstance(mcmc.get("acc_BF", {}), dict) else {}
    acc_mix_noBF = _safe_float(acc_noBF.get("mix"))
    acc_mix_BF = _safe_float(acc_BF.get("mix"))

    # ----- Energy
    E_no = _get(j, "energy", "noBF", default={}) or {}
    E_bf = _get(j, "energy", "withBF", default={}) or {}
    E_no_mean = _safe_float(E_no.get("mean"))
    E_no_se   = _safe_float(E_no.get("se"))
    E_bf_mean = _safe_float(E_bf.get("mean"))
    E_bf_se   = _safe_float(E_bf.get("se"))

    dE = E_bf_mean - E_no_mean
    dE_rel = (dE / E_no_mean) if np.isfinite(E_no_mean) and E_no_mean != 0 else float("nan")
    z = _zscore(dE, E_no_se, E_bf_se)

    sys_energy_mcmc = dict(
        file=path.name,
        N=sysN,
        omega=omega,
        d=d,
        sampler=method,
        sigma_step=step_sigma,
        acc_noBF=acc_mix_noBF,
        acc_BF=acc_mix_BF,
        E_noBF=E_no_mean,
        SE_noBF=E_no_se,
        E_BF=E_bf_mean,
        SE_BF=E_bf_se,
        dE=dE,
        dE_rel=dE_rel,
        z_dE=z,
        winsor_pct=_safe_float(j.get("winsor_pct", float("nan"))),
    )

    # ----- PINN
    pinn = _get(j, "pinn", default={}) or {}
    rho_in_eff = _safe_float(pinn.get("rho_in_pca_eff_rank"))
    feat_eff   = _safe_float(pinn.get("feature_pca_eff_rank"))

    expvar_top8 = _as_list(pinn.get("rho_in_expvar_top8"))
    exp1 = _safe_float(expvar_top8[0]) if len(expvar_top8) > 0 else float("nan")
    exp2 = _safe_float(expvar_top8[1]) if len(expvar_top8) > 1 else float("nan")
    exp3 = _safe_float(expvar_top8[2]) if len(expvar_top8) > 2 else float("nan")
    exp1_3 = exp1 + exp2 + exp3 if (np.isfinite(exp1) and np.isfinite(exp2) and np.isfinite(exp3)) else float("nan")

    probes = pinn.get("probes", {}) if isinstance(pinn.get("probes", {}), dict) else {}
    probe_names = _as_list(probes.get("names"))
    probe_R2 = _as_list(probes.get("R2"))
    probe_map = {str(n): _safe_float(r) for n, r in zip(probe_names, probe_R2, strict=False)}

    branch_ab = pinn.get("branch_ablation", {}) if isinstance(pinn.get("branch_ablation", {}), dict) else {}
    pc1_pow   = pinn.get("pc1_block_power", {}) if isinstance(pinn.get("pc1_block_power", {}), dict) else {}

    pc_ab = pinn.get("pc_ablation", {}) if isinstance(pinn.get("pc_ablation", {}), dict) else {}
    k_list = [int(k) for k in _as_list(pc_ab.get("k"))]
    mae_pc = [float(x) for x in _as_list(pc_ab.get("mae_pc"))]
    mae_rand = [float(x) for x in _as_list(pc_ab.get("mae_rand"))]
    k_10 = _k_threshold(k_list, mae_pc, 0.10)
    k_05 = _k_threshold(k_list, mae_pc, 0.05)

    nf_grad = _as_list(pinn.get("near_field_grad_share"))
    nf_map = {float(r.get("q")): _safe_float(r.get("share")) for r in nf_grad if isinstance(r, dict)}

    efc = pinn.get("energy_feature_corr", {}) if isinstance(pinn.get("energy_feature_corr", {}), dict) else {}
    corr_features = _as_list(efc.get("corr_features"))
    corr_out_base = _safe_float(efc.get("corr_out_base"))
    corr_cusp = _safe_float(efc.get("corr_cusp"))

    rho_w = pinn.get("rho_weights", {}) if isinstance(pinn.get("rho_weights", {}), dict) else {}
    rho_std = _as_list(rho_w.get("std"))
    rho_weff = _as_list(rho_w.get("w_eff"))
    rho_imp = _as_list(rho_w.get("importance"))

    rho_weights_long: list[dict[str, Any]] = []
    for i in range(max(len(rho_std), len(rho_weff), len(rho_imp))):
        rho_weights_long.append(dict(
            file=path.name, N=sysN, omega=omega, idx=i,
            std=_safe_float(rho_std[i]) if i < len(rho_std) else float("nan"),
            w_eff=_safe_float(rho_weff[i]) if i < len(rho_weff) else float("nan"),
            importance=_safe_float(rho_imp[i]) if i < len(rho_imp) else float("nan"),
        ))

    energy_feat_corr_long: list[dict[str, Any]] = []
    for i, c in enumerate(corr_features):
        energy_feat_corr_long.append(dict(file=path.name, N=sysN, omega=omega, idx=i, corr=_safe_float(c)))

    pinn_pc_ablation_long: list[dict[str, Any]] = []
    for k, a, r in zip(k_list, mae_pc, mae_rand, strict=False):
        pinn_pc_ablation_long.append(dict(file=path.name, N=sysN, omega=omega, k=int(k), mae_pc=float(a), mae_rand=float(r)))

    pinn_summary = dict(
        file=path.name, N=sysN, omega=omega,
        effRank_rho_in=rho_in_eff,
        effRank_feat=feat_eff,
        expvar_pc1=exp1,
        expvar_pc2=exp2,
        expvar_pc3=exp3,
        var_PC1_3=exp1_3,
        k_mae10=k_10, k_mae05=k_05,
        R2_rmean=probe_map.get("r_mean", float("nan")),
        R2_rvar=probe_map.get("r_var", float("nan")),
        R2_rmin=probe_map.get("r_min", float("nan")),
        R2_Pr025=probe_map.get("Pr(r<0.25)", float("nan")),
        R2_shell=probe_map.get("shell_contrast", float("nan")),
        abl_phi=_safe_float(branch_ab.get("phi")),
        abl_psi=_safe_float(branch_ab.get("psi")),
        abl_x=_safe_float(branch_ab.get("extras")),
        PC1_phi=_safe_float(pc1_pow.get("phi")),
        PC1_psi=_safe_float(pc1_pow.get("psi")),
        PC1_x=_safe_float(pc1_pow.get("extras")),
        near_q01=nf_map.get(0.01, float("nan")),
        near_q05=nf_map.get(0.05, float("nan")),
        near_q10=nf_map.get(0.10, float("nan")),
        corr_out_base=corr_out_base,
        corr_cusp=corr_cusp,
        corr_feat_absmax=(float(np.nanmax(np.abs(corr_features))) if len(corr_features) else float("nan")),
    )

    # ----- Backflow
    bf = _get(j, "backflow", default={}) or {}
    bf_kind = str(bf.get("bf_kind", ""))

    pca_dx = bf.get("pca_dx", {}) if isinstance(bf.get("pca_dx", {}), dict) else {}
    dx_eff = _safe_float(pca_dx.get("eff_rank"))
    dx_exp8 = _as_list(pca_dx.get("expvar_top8"))
    dx_exp1 = _safe_float(dx_exp8[0]) if len(dx_exp8) > 0 else float("nan")
    dx_exp2 = _safe_float(dx_exp8[1]) if len(dx_exp8) > 1 else float("nan")

    pc_ab_dx = bf.get("pc_ablation_dx", {}) if isinstance(bf.get("pc_ablation_dx", {}), dict) else {}
    k_dx = [int(k) for k in _as_list(pc_ab_dx.get("k"))]
    rel_err_pc = [float(x) for x in _as_list(pc_ab_dx.get("rel_err_pc"))]
    rel_err_rand = [float(x) for x in _as_list(pc_ab_dx.get("rel_err_rand"))]
    dx_k_10 = _k_threshold(k_dx, rel_err_pc, 0.10)

    dx_pc_ablation_long: list[dict[str, Any]] = []
    for k, a, r in zip(k_dx, rel_err_pc, rel_err_rand, strict=False):
        dx_pc_ablation_long.append(dict(file=path.name, N=sysN, omega=omega, k=int(k), rel_err_pc=float(a), rel_err_rand=float(r)))

    nf_dx2 = bf.get("near_field_dx2", {}) if isinstance(bf.get("near_field_dx2", {}), dict) else {}
    nf_dx_rows = _as_list(nf_dx2.get("rows"))
    nf_dx_map = {float(r.get("q")): _safe_float(r.get("share")) for r in nf_dx_rows if isinstance(r, dict)}

    probes_dx = bf.get("probes_dx", {}) if isinstance(bf.get("probes_dx", {}), dict) else {}
    tnames = _as_list(probes_dx.get("target_names"))
    r2s = _as_list(probes_dx.get("R2"))
    dx_probe_map = {str(n): _safe_float(r) for n, r in zip(tnames, r2s, strict=False)}
    decomp = probes_dx.get("pc1_decomposition", {}) if isinstance(probes_dx.get("pc1_decomposition", {}), dict) else {}
    com_frac = _safe_float(decomp.get("com_energy_frac"))
    rel_frac = _safe_float(decomp.get("rel_energy_frac"))
    rad_frac = _safe_float(decomp.get("radial_frac_of_rel"))
    tan_frac = _safe_float(decomp.get("tangential_frac_of_rel"))
    per_particle_share = _as_list(decomp.get("per_particle_share"))

    dx_pc1_particle_long: list[dict[str, Any]] = []
    for i, s in enumerate(per_particle_share):
        dx_pc1_particle_long.append(dict(file=path.name, N=sysN, omega=omega, particle=i, share=_safe_float(s)))

    ctnn_internal = bf.get("ctnn_internal", {}) if isinstance(bf.get("ctnn_internal", {}), dict) else {}
    ctnn_note = str(ctnn_internal.get("note", ""))

    backflow_summary = dict(
        file=path.name, N=sysN, omega=omega,
        BF=bf_kind,
        effRank_dx=dx_eff,
        expvar1=dx_exp1,
        expvar2=dx_exp2,
        k_err10=dx_k_10,
        near_dx_q05=nf_dx_map.get(0.05, float("nan")),
        dx_R2_rmean=dx_probe_map.get("r_mean", float("nan")),
        dx_R2_Pr025=dx_probe_map.get("Pr(r<0.25)", float("nan")),
        PC1_COM=com_frac,
        PC1_rel=_safe_float(rel_frac),
        PC1_radial=rad_frac,
        PC1_tangential=tan_frac,
        note=ctnn_note,
    )

    return Extracted(
        system_energy_mcmc=sys_energy_mcmc,
        pinn_summary=pinn_summary,
        backflow_summary=backflow_summary,
        rho_weights_long=rho_weights_long,
        energy_feat_corr_long=energy_feat_corr_long,
        dx_pc1_particle_long=dx_pc1_particle_long,
        pinn_pc_ablation_long=pinn_pc_ablation_long,
        dx_pc_ablation_long=dx_pc_ablation_long,
    )


def collect_all(in_dir: Path) -> dict[str, pd.DataFrame]:
    # Match your notebook naming exactly
    files = sorted(in_dir.glob("analysis_CTNN_N*_omega_*.json"))
    if not files:
        raise FileNotFoundError(f"No files found matching analysis_CTNN_N*_omega_*.json in {in_dir}")

    rows_sys, rows_pinn, rows_bf = [], [], []
    long_rho_w, long_efc, long_dx_pc1 = [], [], []
    long_pinn_ab, long_dx_ab = [], []

    for p in files:
        ex = _extract_one(p)
        rows_sys.append(ex.system_energy_mcmc)
        rows_pinn.append(ex.pinn_summary)
        rows_bf.append(ex.backflow_summary)
        long_rho_w.extend(ex.rho_weights_long)
        long_efc.extend(ex.energy_feat_corr_long)
        long_dx_pc1.extend(ex.dx_pc1_particle_long)
        long_pinn_ab.extend(ex.pinn_pc_ablation_long)
        long_dx_ab.extend(ex.dx_pc_ablation_long)

    df_sys  = pd.DataFrame(rows_sys).sort_values(["N","omega"])
    df_pinn = pd.DataFrame(rows_pinn).sort_values(["N","omega"])
    df_bf   = pd.DataFrame(rows_bf).sort_values(["N","omega"])

    return {
        "system_energy_mcmc": df_sys,
        "pinn_summary": df_pinn,
        "backflow_summary": df_bf,
        "rho_weights_long": pd.DataFrame(long_rho_w).sort_values(["N","omega","idx"]),
        "energy_feature_corr_long": pd.DataFrame(long_efc).sort_values(["N","omega","idx"]),
        "dx_pc1_per_particle_long": pd.DataFrame(long_dx_pc1).sort_values(["N","omega","particle"]),
        "pinn_pc_ablation_long": pd.DataFrame(long_pinn_ab).sort_values(["N","omega","k"]),
        "dx_pc_ablation_long": pd.DataFrame(long_dx_ab).sort_values(["N","omega","k"]),
    }


# ---------------------------
# Run + write categorized tables
# ---------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)
dfs = collect_all(IN_DIR)

# Make omega string in the compact tables
for key in ("system_energy_mcmc","pinn_summary","backflow_summary"):
    if "omega" in dfs[key].columns:
        dfs[key] = dfs[key].copy()
        dfs[key]["omega"] = dfs[key]["omega"].map(_fmt_omega_str)

# --- TABLE 1: System + sampling + energy
df1 = dfs["system_energy_mcmc"].copy()
keep1 = ["N","omega","sampler","sigma_step","acc_noBF","acc_BF","E_noBF","SE_noBF","E_BF","SE_BF","dE","z_dE"]
df1 = df1[[c for c in keep1 if c in df1.columns]]

_write_df(
    df1,
    OUT_DIR/"table_ctnn_system_energy_mcmc.csv",
    OUT_DIR/"table_ctnn_system_energy_mcmc.tex",
    caption="Sampling and energy diagnostics for CTNN analyses. The combined significance indicator is $z=\\Delta E/\\sqrt{\\mathrm{SE}_{\\mathrm{noBF}}^2+\\mathrm{SE}_{\\mathrm{BF}}^2}$.",
    label="tab:ctnn_system_energy_mcmc",
)

# --- TABLE 2: PINN (low-rank + interpretability + block importance)
df2 = dfs["pinn_summary"].copy()
keep2 = [
    "N","omega",
    "effRank_rho_in","var_PC1_3","k_mae10","k_mae05",
    "R2_rmean","R2_rvar","R2_Pr025","R2_shell",
    "abl_phi","abl_psi","abl_x",
    "PC1_phi","PC1_psi","PC1_x",
    "near_q05",
]
df2 = df2[[c for c in keep2 if c in df2.columns]]

# shorten a few column names (avoid huge headers)
df2 = _short_columns(df2, {
    "effRank_rho_in": "effRank(ρ_in)",
    "var_PC1_3": "var(PC1–3)",
    "k_mae10": "k@10%",
    "k_mae05": "k@5%",
    "R2_rmean": "R²(r̄)",
    "R2_rvar": "R²(var r)",
    "R2_Pr025": "R²(Pr r<0.25)",
    "R2_shell": "R²(shell)",
    "abl_phi": "abl(φ)",
    "abl_psi": "abl(ψ)",
    "abl_x": "abl(x)",
    "PC1_phi": "PC1φ",
    "PC1_psi": "PC1ψ",
    "PC1_x": "PC1x",
    "near_q05": "near(5%)",
})

_write_df(
    df2,
    OUT_DIR/"table_ctnn_pinn_summary.csv",
    OUT_DIR/"table_ctnn_pinn_summary.tex",
    caption="PINN summary: low-rank structure, functional compressibility (PC-ablation), and linear interpretability ($R^2$ against simple geometric observables).",
    label="tab:ctnn_pinn_summary",
)

# --- TABLE 3: Backflow dx (complexity + structure + near-field)
df3 = dfs["backflow_summary"].copy()
keep3 = [
    "N","omega","BF",
    "effRank_dx","expvar1","expvar2","k_err10",
    "PC1_COM","PC1_radial","PC1_tangential",
    "near_dx_q05",
    "dx_R2_rmean","dx_R2_Pr025",
    "note",
]
df3 = df3[[c for c in keep3 if c in df3.columns]]

df3 = _short_columns(df3, {
    "effRank_dx": "effRank(dx)",
    "expvar1": "expvar1",
    "expvar2": "expvar2",
    "k_err10": "k@10%",
    "PC1_COM": "COM",
    "PC1_radial": "radial",
    "PC1_tangential": "tangential",
    "near_dx_q05": "near(5%)",
    "dx_R2_rmean": "R²_dx(r̄)",
    "dx_R2_Pr025": "R²_dx(Pr)",
})

_write_df(
    df3,
    OUT_DIR/"table_ctnn_backflow_dx_summary.csv",
    OUT_DIR/"table_ctnn_backflow_dx_summary.tex",
    caption="Backflow summary: $\\Delta x$ complexity (effective rank), sensitivity/compressibility (k@10\\% relative error), and structural decomposition of PC1 into COM vs relative motion and radial vs tangential components.",
    label="tab:ctnn_backflow_dx_summary",
)

# --- LONG tables (not wide; good for plotting/appendix)
for name, df in dfs.items():
    (OUT_DIR/f"{name}.csv").write_text(df.to_csv(index=False), encoding="utf-8")

print("[ok] scanned:", IN_DIR.resolve())
print("[ok] wrote  :", OUT_DIR.resolve())
print("TeX tables:")
print("  -", (OUT_DIR/"table_ctnn_system_energy_mcmc.tex").name)
print("  -", (OUT_DIR/"table_ctnn_pinn_summary.tex").name)
print("  -", (OUT_DIR/"table_ctnn_backflow_dx_summary.tex").name)
print("Long CSVs:")
for nm in ["rho_weights_long","energy_feature_corr_long","dx_pc1_per_particle_long","pinn_pc_ablation_long","dx_pc_ablation_long"]:
    print("  -", f"{nm}.csv")



[ok] scanned: /Users/aleksandersekkelsten/thesis/results/tables/jsons
[ok] wrote  : /Users/aleksandersekkelsten/thesis/results/tables/texfiles
TeX tables:
  - table_ctnn_system_energy_mcmc.tex
  - table_ctnn_pinn_summary.tex
  - table_ctnn_backflow_dx_summary.tex
Long CSVs:
  - rho_weights_long.csv
  - energy_feature_corr_long.csv
  - dx_pc1_per_particle_long.csv
  - pinn_pc_ablation_long.csv
  - dx_pc_ablation_long.csv


In [1]:
# %% ============================================================
# JSON -> CSV (dimensional + non-MCMC metrics)
# Scans ../results/tables/newest for:
#   N{N}_omega{omega}.json (and duplicates like " (1)")
# Targets:
#   N in {2,6,12,20}
#   omega in {0.001, 0.01, 0.1, 0.5, 1.0}
# ============================================================

from __future__ import annotations

import re
from dataclasses import dataclass
from pathlib import Path

BASE_DIR = Path("../results/tables/newest")
OUT_CSV  = BASE_DIR / "dimensional_summary_N2_6_12_20_omegas_new.csv"

TARGET_N = [2, 6, 12, 20]
TARGET_OMEGA = [0.001, 0.01, 0.1, 0.5, 1.0]

# If your JSON uses a dedicated "dimensional results" section, list candidate keys here.
# If found, we take ONLY that subtree (minus MCMC-ish subkeys).
DIMENSIONAL_ROOT_CANDIDATES = [
    "dimensional_results", "dimensional", "dimension_results", "dims", "dimensions",
    "repdim", "repdim_suite", "repdim_suite_results", "suite", "results"
]

# Exclusion patterns: anything matching is removed (recursively) from output.
EXCLUDE_KEY_PATTERNS = [
    r"\bmcmc\b",
    r"\bmcmc_stats\b",
    r"\bsampler\b",
    r"\baccept\b", r"\bacceptance\b", r"\bacc\b",
    r"\bn_steps\b", r"\bsteps\b", r"\bn_samples\b", r"\bn_samples_effective\b",
    r"\bess\b", r"\bautocorr\b", r"\btau\b",
    r"\bburn\b", r"\bburnin\b", r"\bthin\b",
    r"\bseed\b", r"\brng\b",
    r"\btime_per_step\b", r"\bruntime\b",
]

# Regex to parse filenames like:
#   N2_omega0.001.json
#   N2_omega0.001 (1).json
# also tolerates omega_0.001
FNAME_RE = re.compile(
    r"N(?P<N>\d+)_omega_?(?P<omega>[0-9]+(?:\.[0-9]+)?(?:e[+-]?\d+)?)",
    re.IGNORECASE
)

OMEGA_TOL = 1e-12


In [2]:
# %% ============================================================
# Helpers: parsing, pruning, flattening
# ============================================================

def _read_json(p: Path) -> dict[str, Any]:
    with p.open("r", encoding="utf-8") as f:
        return json.load(f)

def _is_excluded_key(key: str) -> bool:
    k = key.lower()
    for pat in EXCLUDE_KEY_PATTERNS:
        if re.search(pat, k):
            return True
    return False

def _prune_excluded(obj: Any) -> Any:
    """
    Recursively remove dict entries whose key matches EXCLUDE_KEY_PATTERNS.
    """
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            if _is_excluded_key(str(k)):
                continue
            out[k] = _prune_excluded(v)
        return out
    elif isinstance(obj, list):
        return [_prune_excluded(x) for x in obj]
    else:
        return obj

def _select_dimensional_root(d: dict[str, Any]) -> dict[str, Any]:
    """
    If we can find a clear "dimensional results" subtree, use it; otherwise
    use the whole dict and just prune MCMC-ish keys.
    """
    for k in DIMENSIONAL_ROOT_CANDIDATES:
        if k in d and isinstance(d[k], dict):
            return d[k]
    return d

def _normalize_value(v: Any, max_list_len: int = 20, max_str_len: int = 2000) -> Any:
    """
    Convert non-scalar values into something CSV-safe.
    - Scalars: keep
    - Short lists: join by ';' if scalarish
    - Otherwise: JSON-dump (truncated) or mark omitted
    """
    if v is None:
        return None
    if isinstance(v, (bool, int, float, str)):
        if isinstance(v, float) and (math.isnan(v) or math.isinf(v)):
            return str(v)
        return v

    # numpy scalars sometimes appear; handle gently
    try:
        import numpy as np
        if isinstance(v, (np.floating, np.integer, np.bool_)):
            vv = v.item()
            return _normalize_value(vv, max_list_len=max_list_len, max_str_len=max_str_len)
    except Exception:
        pass

    if isinstance(v, (list, tuple)):
        if len(v) <= max_list_len and all(isinstance(x, (bool, int, float, str)) or x is None for x in v):
            return ";".join("" if x is None else str(x) for x in v)
        s = json.dumps(v, ensure_ascii=False)
        if len(s) <= max_str_len:
            return s
        return "[omitted_large_list]"

    if isinstance(v, dict):
        s = json.dumps(v, ensure_ascii=False)
        if len(s) <= max_str_len:
            return s
        return "[omitted_large_dict]"

    return str(v)

def _flatten_dict(d: dict[str, Any], prefix: str = "") -> dict[str, Any]:
    """
    Flatten nested dictionaries into dot-separated keys.
    Lists are not expanded to many columns; they are normalized into strings.
    """
    out: dict[str, Any] = {}
    for k, v in d.items():
        kk = f"{prefix}.{k}" if prefix else str(k)
        if isinstance(v, dict):
            out.update(_flatten_dict(v, kk))
        else:
            out[kk] = _normalize_value(v)
    return out

def _closest_target_omega(omega: float) -> float | None:
    for t in TARGET_OMEGA:
        if abs(omega - t) <= OMEGA_TOL:
            return t
    return None


In [3]:
# %% ============================================================
# 1) Scan directory and map (N, omega) -> candidate files
# ============================================================

if not BASE_DIR.exists():
    raise FileNotFoundError(f"BASE_DIR does not exist: {BASE_DIR.resolve()}")

all_json = sorted(BASE_DIR.glob("*.json"))

candidates: dict[tuple[int, float], list[Path]] = {}
unmatched: list[Path] = []

for p in all_json:
    m = FNAME_RE.search(p.stem)
    if not m:
        unmatched.append(p)
        continue

    N = int(m.group("N"))
    if N not in TARGET_N:
        continue

    try:
        omega_val = float(m.group("omega"))
    except ValueError:
        unmatched.append(p)
        continue

    omega_t = _closest_target_omega(omega_val)
    if omega_t is None:
        # Not one of the target omegas
        continue

    candidates.setdefault((N, omega_t), []).append(p)

# choose newest file if multiple
chosen: dict[tuple[int, float], Path] = {}
duplicates: dict[tuple[int, float], list[Path]] = {}

for key, paths in candidates.items():
    if len(paths) == 1:
        chosen[key] = paths[0]
    else:
        # Pick by modification time (newest)
        paths_sorted = sorted(paths, key=lambda x: x.stat().st_mtime, reverse=True)
        chosen[key] = paths_sorted[0]
        duplicates[key] = paths_sorted

# compute missing combos
missing: list[tuple[int, float]] = []
for N in TARGET_N:
    for om in TARGET_OMEGA:
        if (N, om) not in chosen:
            missing.append((N, om))

print(f"[scan] BASE_DIR: {BASE_DIR.resolve()}")
print(f"[scan] total .json files: {len(all_json)}")
print(f"[scan] matched target combos: {len(chosen)} / {len(TARGET_N)*len(TARGET_OMEGA)}")
print(f"[scan] missing combos: {len(missing)}")
print(f"[scan] duplicates (kept newest): {len(duplicates)}")
if unmatched:
    print(f"[scan] filename-unmatched jsons (ignored): {len(unmatched)} (example: {unmatched[0].name})")


[scan] BASE_DIR: /Users/aleksandersekkelsten/thesis/results/tables/newest
[scan] total .json files: 20
[scan] matched target combos: 20 / 20
[scan] missing combos: 0
[scan] duplicates (kept newest): 0


In [4]:
# %% ============================================================
# 2) Parse JSONs, drop MCMC statistics, flatten, build DataFrame, write CSV
# ============================================================

rows: list[dict[str, Any]] = []

for (N, omega), path in sorted(chosen.items(), key=lambda x: (x[0][0], x[0][1])):
    try:
        raw = _read_json(path)
    except Exception as e:
        rows.append({
            "N": N,
            "omega": omega,
            "file": str(path),
            "file_chosen_reason": "read_error",
            "read_error": repr(e),
        })
        continue

    # Prefer a dedicated "dimensional results" subtree if it exists; else whole dict
    subtree = _select_dimensional_root(raw)
    subtree = _prune_excluded(subtree)

    flat = _flatten_dict(subtree)

    # Add metadata columns first
    r: dict[str, Any] = {
        "N": N,
        "omega": omega,
        "file": str(path),
        "file_mtime": path.stat().st_mtime,
        "file_chosen_reason": "newest_mtime" if (N, omega) in duplicates else "unique",
    }
    r.update(flat)
    rows.append(r)

df = pd.DataFrame(rows)

# Sort columns: metadata first, then metrics
meta_cols = ["N", "omega", "file", "file_mtime", "file_chosen_reason", "read_error"]
meta_cols = [c for c in meta_cols if c in df.columns]
other_cols = sorted([c for c in df.columns if c not in meta_cols])
df = df[meta_cols + other_cols].sort_values(["N", "omega"], ascending=[True, True])

df.to_csv(OUT_CSV, index=False)
print(f"[write] CSV written to: {OUT_CSV.resolve()}")
print(f"[write] rows: {len(df)} | cols: {len(df.columns)}")

display(df.head(10))


[write] CSV written to: /Users/aleksandersekkelsten/thesis/results/tables/newest/dimensional_summary_N2_6_12_20_omegas_new.csv
[write] rows: 20 | cols: 376


,N,omega,file,file_mtime,file_chosen_reason,alignments.bf_keys,alignments.present_keys_total,alignments.rho_keys,diagnostics.checkpoint_search.basis_dims.n_occ,diagnostics.checkpoint_search.basis_dims.nx,...,repgeom.node_space.f_net.phi.mean_norm,repgeom.node_space.f_net.phi.n,repgeom.node_space.f_net.phi.phys_corr_pc1,repgeom.node_space.f_net.phi.phys_corr_rep_norm,repgeom.node_space.f_net.phi.sv_top,repgeom.node_space.f_net.phi.var_trace,repgeom.summaries.edge.n_modules,repgeom.summaries.global.n_modules,repgeom.summaries.node.n_modules,traceback
0,2,0.001,../results/tables/newest/N2_omega0.001.json,1.768821e+09,unique,NaN,NaN,NaN,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Traceback (most recent call last):\n File ""/t..."
1,2,0.010,../results/tables/newest/N2_omega0.01.json,1.768821e+09,unique,NaN,NaN,NaN,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Traceback (most recent call last):\n File ""/t..."
2,2,0.100,../results/tables/newest/N2_omega0.1.json,1.768821e+09,unique,NaN,NaN,NaN,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Traceback (most recent call last):\n File ""/t..."
3,2,0.500,../results/tables/newest/N2_omega0.5.json,1.768821e+09,unique,NaN,NaN,NaN,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Traceback (most recent call last):\n File ""/t..."
4,2,1.000,../results/tables/newest/N2_omega1.0.json,1.768821e+09,unique,NaN,NaN,NaN,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Traceback (most recent call last):\n File ""/t..."
5,6,0.001,../results/tables/newest/N6_omega0.001 2.json,1.768821e+09,unique,NaN,NaN,NaN,3,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Traceback (most recent call last):\n File ""/t..."
6,6,0.010,../results/tables/newest/N6_omega0.01.json,1.768821e+09,unique,NaN,NaN,NaN,3,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Traceback (most recent call last):\n File ""/t..."
7,6,0.100,../results/tables/newest/N6_omega0.1.json,1.768821e+09,unique,NaN,NaN,NaN,3,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Traceback (most recent call last):\n File ""/t..."
8,6,0.500,../results/tables/newest/N6_omega0.5.json,1.768821e+09,unique,NaN,NaN,NaN,3,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Traceback (most recent call last):\n File ""/t..."
9,6,1.000,../results/tables/newest/N6_omega1.0 2.json,1.768821e+09,unique,NaN,NaN,NaN,3,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Traceback (most recent call last):\n File ""/t..."
